In [127]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate, SystemMessagePromptTemplate, MessagesPlaceholder, ChatPromptTemplate
from langchain_openrouter import ChatOpenRouter
from pydantic import BaseModel, Field
from openrouter_embedding import CustomEmbedOpenRouter
from typing import List
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import chromadb
import os

In [125]:
load_dotenv()

True

In [11]:
apikey = "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"

In [75]:
system_prompt = SystemMessagePromptTemplate.from_template(""" Tu es un assistant juriste specialie dans le droit civil.

Ton objectif est de repondre aux questions des clients en te basant sur le code civil.

IMPORTANT
Ne fais pas de supposition et n'invente RIEN qui n'est pas ecris dans le code civil.
toutes tes reponses doivent etres sourcees et contenir le ou les articles du code civil auxquels tu fais reference dans ta reponse.

Pour repondre aux questions des clients, il faut que tu reformule leurs questionsdans un langage qui puisse faciliter 
l'obtention des articles pertinents.

""")

In [76]:
prompt_few_shot = PromptTemplate(template="Client : {ex}\nToi : {ans}\n",input_variables=["ex","ans"])


In [81]:
# METTRE DES VRAIS EXEMPLES
ex = [
  { "ex": "What is the capital of France?", "ans": "Paris" },
  { "ex": "What is the capital of Japan?", "ans": "Tokyo" },
]


In [78]:
print(prompt_few_shot.format(**ex[0]))

Client : What is the capital of France?
Toi : Paris



In [84]:
few_shot_prompt = FewShotPromptTemplate(
    examples=ex,
    example_prompt=prompt_few_shot,
    prefix="Voici quelques exemples de raisonnements attendus pour repondre a un client :",
    suffix="Client question : {input}",
    input_variables=["input"],
    example_separator="\n",
)

In [85]:
formatted_prompt = few_shot_prompt.format(input="En tant que citoyen francais, ai-je le droit de tuer ?")
print(formatted_prompt)

Voici quelques exemples de raisonnements attendus pour repondre a un client :
Client : What is the capital of France?
Toi : Paris

Client : What is the capital of Japan?
Toi : Tokyo

Client question : En tant que citoyen francais, ai-je le droit de tuer ?


In [89]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful career coach."),
   # MessagesPlaceholder("conversation"), 
    ("human", "{current_question}")
])


In [90]:
model = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free",api_key=apikey)

In [94]:

chain = chat_prompt | model | StrOutputParser()

In [95]:
chain.invoke({"current_question":"I am a new graduate in AI and data science and I can't find a job, what to do ?"})

'This is brutally common right now. The **2024–2025 entry-level market is the toughest it’s been in a decade** due to a mix of hiring freezes, senior layoffs flooding the market with experience, AI automation of junior tasks, and an explosion of CS/DS graduates.\n\n**You are not "bad" at this. You are hitting a structural market wall.**\n\nHere is a tactical, no-fluff battle plan divided into **Immediate Triage (Week 1–2)**, **Strategic Positioning (Month 1)**, and **Survival/Long Game (Month 3+)**.\n\n---\n\n### 🚨 PHASE 1: IMMEDIATE TRIAGE (Do this *this week*)\n\n#### 1. Audit your "Paper Trail" (Resume + LinkedIn + GitHub)\nRecruiters spend **6–10 seconds** scanning. If you don\'t pass the "Ctrl+F" test for keywords, you are auto-rejected.\n*   **Tailor per application:** Keep a "Master Resume," but submit a **Targeted Resume** for every single role. Match the JD keywords (e.g., if they say "PyTorch," don\'t just say "Deep Learning frameworks").\n*   **Kill the "Academic" vibe:** Re

In [99]:
print('This is brutally common right now. The **2024–2025 entry-level market is the toughest it’s been in a decade** due to a mix of hiring freezes, senior layoffs flooding the market with experience, AI automation of junior tasks, and an explosion of CS/DS graduates.\n\n**You are not "bad" at this. You are hitting a structural market wall.**\n\nHere is a tactical, no-fluff battle plan divided into **Immediate Triage (Week 1–2)**, **Strategic Positioning (Month 1)**, and **Survival/Long Game (Month 3+)**.\n\n---\n\n### 🚨 PHASE 1: IMMEDIATE TRIAGE (Do this *this week*)\n\n#### 1. Audit your "Paper Trail" (Resume + LinkedIn + GitHub)\nRecruiters spend **6–10 seconds** scanning. If you don\'t pass the "Ctrl+F" test for keywords, you are auto-rejected.\n*   **Tailor per application:** Keep a "Master Resume," but submit a **Targeted Resume** for every single role. Match the JD keywords (e.g., if they say "PyTorch," don\'t just say "Deep Learning frameworks").\n*   **Kill the "Academic" vibe:** Remove "Coursework," "Dean\'s List," "GPA" (unless >3.8). Replace with **Projects** and **Skills**.\n*   **The "So What?" Test:** Every bullet point must answer: *What did you do? What tool? What was the result/impact?*\n    *   ❌ *Built a classification model for fraud detection.*\n    *   ✅ *Engineered an XGBoost fraud detection pipeline (Python/SQL) on 2M transactions, improving recall by 15% over baseline, deployed via Flask/Docker on AWS EC2.*\n*   **GitHub/Portfolio:** **Must be public.** 3–5 polished repos max. Each needs: `README` (problem, data, architecture, results, *how to run it*), clean code (modular, `requirements.txt`), and a notebook/demo. **One great end-to-end project > ten messy Jupyter notebooks.**\n\n#### 2. Fix the "Application Funnel" Math\nIf you applied to 200 jobs and heard nothing, your **Top of Funnel** is broken.\n*   **Target:** 50–100 *tailored* applications/week (yes, it’s a full-time job).\n*   **Track it:** Use a Notion/Excel sheet: `Company | Role | Link | Date Applied | Contact (Recruiter/HM) | Status | Follow-up Date`.\n*   **Stop "Easy Apply" only:** It puts you in a pile of 2,000. Apply via **Company Career Portal** + **Find the Hiring Manager/Recruiter on LinkedIn** + **Send a connection request with a note.**\n\n#### 3. The "Hidden" Application Hack\n> **Find the Hiring Manager (HM) or a Senior DS on the team → Connect on LinkedIn with a 300-char note.**\n> *Script:* *"Hi [Name], I just applied for the [Role ID] role. I\'ve built [specific relevant project: e.g., \'a RAG pipeline with LlamaIndex\'] which aligns with your work on [mention their recent blog/post/tech stack]. Would value 2 mins of your time to ask what success looks like in this role. Thanks!"*\n*   **Why:** Referrals bypass ATS filters. Even if they don\'t refer, they see your name.\n\n---\n\n### 🎯 PHASE 2: STRATEGIC POSITIONING (Month 1 — Differentiate)\n\n**The market is saturated with "Generalist Junior DS." You must niche down.**\n\n#### 1. Pick a "Wedge" (Choose ONE to market yourself as *right now*)\nDon\'t say "I do AI/ML/DL/NLP/CV/MLOps." Pick the intersection of **Your Strength** + **Market Demand**.\n*   **MLOps/ML Engineer:** *Highest demand for juniors.* Learn: Docker, CI/CD (GitHub Actions), Cloud (AWS/GCP/Azure - pick one), Airflow/Prefect, MLflow, Monitoring (Evidently/Prometheus). **Deploy something.**\n*   **LLM/RAG Application Engineer:** *Hottest niche.* Learn: LangChain/LlamaIndex, Vector DBs (Pinecone/Weaviate/Chroma), Prompt Engineering, Evaluation (RAGAS), Fine-tuning (LoRA/QLoRA). **Build a RAG app on private PDFs.**\n*   **Data/Analytics Engineer (The "Backdoor"):** *Easier entry.* SQL mastery, dbt, Snowflake/BigQuery, Airflow, Data Modeling. Many DS roles are actually 80% SQL/ETL. This gets you *in* the company for internal transfer later.\n*   **Domain-Specific DS:** Finance (Time series/Risk), Bio/Health (Imaging/Genomics), Geo/Satellite, Marketing (Causal Inference/AB Testing). **Leverage any non-DS background you have.**\n\n#### 2. Build *One* "Flagship" Project for your Wedge\n*   **Must be End-to-End:** Data ingestion → Processing → Model/Training → **Deployment (API + UI)** → Monitoring/Logging.\n*   **Use Real/ Messy Data:** Not clean Kaggle CSVs. Use APIs (Reddit, Financial, Weather), scrape (ethically), or use messy public gov datasets.\n*   **Document it like a Senior:** Write a **Technical Blog Post** (Medium/Dev.to/Substack/Personal Site) explaining *why* you chose X over Y, what failed, and what you\'d do in production. **Share this link in every application.**\n\n#### 3. Upskill *Surgically* (Don\'t do another Coursera cert)\n*   **SQL:** Do **LeetCode Hard / StrataScratch / Mode Analytics tutorials** until you can write Window Functions, CTEs, and Query Optimization in your sleep. This is the #1 filter.\n*   **System Design (ML):** Study "Designing ML Systems" (Chip Huyen) / "ML System Design" interviews. Know the difference between Batch vs. Online inference, Feature Stores, Data Drift.\n*   **Software Engineering Hygiene:** Git (branching/PRs), Testing (pytest), Typing (mypy), Docker, Virtual Envs. **Juniors who write clean, tested, containerized code get hired over PhDs who only notebook.**\n\n---\n\n### 🤝 PHASE 3: THE HUMAN ELEMENT (Networking & Mindset)\n\n#### 1. Informational Interviews > Cold Applications\n*   Target: **Alumni (LinkedIn "Alumni" tool)**, **2nd-degree connections**, **People at target companies**.\n*   Ask: *"What does your day look like?" "What skills is your team missing?" "What would you learn if you were me?"*\n*   **Never ask for a job.** Ask for *insight*. They often *become* the referral.\n\n#### 2. Volunteer / Fractional / Freelance (Get "Real" Experience)\n*   **Omdena / DataKind / Zindi / Kaggle Competitions (join teams):** Collaborative projects mimic work environments.\n*   **Non-profits / Local Small Biz:** "I\'ll build you a churn prediction dashboard / automate this report for free/cheap in exchange for a testimonial and permission to share the sanitized case study."\n*   **Upwork/Fiverr (Low pay, high reps):** Do 5 small $50–$100 gigs (data cleaning, dashboard, simple model). Puts "Freelance ML Engineer" on resume + client reviews.\n\n#### 3. Consider the "Stepping Stone" Roles\nDon\'t hold out *only* for "Data Scientist" or "ML Engineer."\n*   **Data Analyst (Heavy SQL/Python):** Easier to get. Build internal brand → Transfer in 12–18 months.\n*   **MLOps / Data Engineer / Research Assistant / Applied Scientist Intern (Post-grad):** Many companies hire "New Grad" interns that convert to FTE.\n*   **Sales Engineer / Solutions Engineer (AI/DS focus):** High pay, technical, great path to PM or Eng.\n\n---\n\n### 🛡️ PHASE 4: SURVIVAL & MENTAL HEALTH\n\n1.  **Structure your days:** 9–12: Applications/Outreach. 1–4: Building/Upskilling. 4–5: Networking/LinkedIn. **Treat it like a 9-5.**\n2.  **Rejection is data, not judgment.** Track *where* you fail:\n    *   No reply → Resume/Keywords/Volume issue.\n    *   Recruiter screen fail → Storytelling/Behavioral prep.\n    *   Technical screen fail → Coding/SQL/ML Theory gaps.\n    *   Onsite fail → System Design/Communication/Depth.\n    *   *Fix the specific leak.*\n3.  **Financial Runway:** If money is tight, **take a non-tech job (barista, tutoring, admin, manual labor) part-time.** It removes desperation from interviews. Desperation smells; confidence sells.\n4.  **Community:** Join a **job-hunting accountability group** (Discord/Slack: Data Science Job Seekers, Tech Interview Prep, local meetups). Misery loves company; accountability creates action.\n\n---\n\n### 📋 YOUR ACTION LIST FOR TODAY\n\n1.  [ ] **Rewrite 1 resume bullet** using the "Action + Tool + Result" formula.\n2.  [ ] **Identify your "Wedge"** (MLOps? RAG? Analytics Eng? Domain?).\n3.  [ ] **Find 3 alumni** at companies you like → Send connection notes.\n4.  [ ] **Apply to 5 roles via Company Portal** + Find HM/Recruiter → Send LinkedIn note.\n5.  [ ] **Write 1 SQL query** (Window function/CTE) or **Dockerize a past project**.\n\n---\n\n### A Final Reality Check\n**It takes the average new grad 4–8 months right now.** Some take 12+.\nYou are playing a **probability game**. Increase your surface area (applications + network + visible proof of skill) and decrease your cycle time (feedback loops on interviews).\n\n**You only need *one* "Yes."**\n\n*Which "Wedge" feels most natural to you? If you tell me your background (thesis, internships, preferred stack), I can help you sharpen the positioning.*')

This is brutally common right now. The **2024–2025 entry-level market is the toughest it’s been in a decade** due to a mix of hiring freezes, senior layoffs flooding the market with experience, AI automation of junior tasks, and an explosion of CS/DS graduates.

**You are not "bad" at this. You are hitting a structural market wall.**

Here is a tactical, no-fluff battle plan divided into **Immediate Triage (Week 1–2)**, **Strategic Positioning (Month 1)**, and **Survival/Long Game (Month 3+)**.

---

### 🚨 PHASE 1: IMMEDIATE TRIAGE (Do this *this week*)

#### 1. Audit your "Paper Trail" (Resume + LinkedIn + GitHub)
Recruiters spend **6–10 seconds** scanning. If you don't pass the "Ctrl+F" test for keywords, you are auto-rejected.
*   **Tailor per application:** Keep a "Master Resume," but submit a **Targeted Resume** for every single role. Match the JD keywords (e.g., if they say "PyTorch," don't just say "Deep Learning frameworks").
*   **Kill the "Academic" vibe:** Remove "Coursework

In [107]:

client = chromadb.HttpClient(host="localhost", port=8000, ssl=False)

In [109]:
collections = client.list_collections()

In [110]:
collection

[Collection(name=articles_code_civil)]

In [111]:
collection = client.get_collection("articles_code_civil")

In [122]:
similars = collection.query(query_embeddings=[queryEmb],n_results=4)

In [123]:
similars

{'ids': [['7b4e89db-3e4d-4e89-8390-4533988d38c8',
   'b4695e2c-41d8-49b0-93e2-14c9d4f34229',
   'c7fab72f-2472-4e4c-9d50-696adfb437ca',
   '33cd695d-76ba-4716-a746-8fa698330764']],
 'distances': [[1.0344296, 1.0722449, 1.0924016, 1.1053252]],
 'embeddings': None,
 'metadatas': [[{'source': 'Article  2278 du code civil'},
   {'source': 'Article  1966 du code civil'},
   {'source': 'Article  494-5 du code civil'},
   {'source': 'Article  544 du code civil'}]],
 'documents': [["\nLa possession est protégée, sans avoir égard au fond du droit, contre le trouble qui l'affecte ou la menace.\n\nLa protection possessoire est pareillement accordée au détenteur contre tout autre que celui de qui il tient ses droits.\n\n## Livre IV : Des sûretés\n\n",
   "\nLes jeux propres à exercer au fait des armes, les courses à pied ou à cheval, les courses de chariot, le jeu de paume et autres jeux de même nature qui tiennent à l'adresse et à l'exercice du corps, sont exceptés de la disposition précédente.\n

In [ ]:
chain = chat_prompt | model | StrOutputParser() | 

In [115]:
customEmb = CustomEmbedOpenRouter()

In [117]:
queryEmb = customEmb.embed_query("Ais-je le droit de detenir des armes ?")

In [128]:
class ArticlesQueries(BaseModel):
    query1 : str = Field(description="Rephrasing of client question to query the vector database")
    query2 : str = Field(description="Second rephrasing of client question")
    query3 : str = Field(description="Third rephrasing of client question")
    keywords : List[str] = Field(description="List of keywords relevant to the client question to find better matches in the vector database")
    

In [133]:
structured_model = model.with_structured_output(ArticlesQueries)

In [134]:
structured_model.invoke("\n\nClient question :\n\nAis-je le droit de porter une arme en france ?")

ArticlesQueries(query1="droit de porter une arme en France réglementation port d'arme", query2="législation française sur le port d'arme conditions autorisation", query3='armes autorisées port public France loi armes', keywords=["port d'arme", 'France', 'législation', 'autorisation', 'réglementation', 'arme à feu', 'arme blanche'])

In [ ]:
new_chain = chain = chat_prompt | structured_model | StrOutputParser() | 